In [2]:
import torch
import numpy as np
import os
from util.dataloader import NSD_proto_multiple
from nilearn import datasets
from nilearn import plotting
from nilearn.surface import load_surf_mesh
from matplotlib.colors import ListedColormap
from matplotlib import pyplot as plt
import cortex
from scipy.optimize import linear_sum_assignment
from matplotlib.lines import Line2D

/home/shichen/anaconda3/envs/torch/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [ ]:
proto_dir = './log_prototype_coco_4_1_1000_128_slim/nsd_proto/'
mri_dir = '/mnt/data/UserData/shichen/data/NSD_dataset/'
subject = 'subj01'
hemi = 'rh'
dataset = NSD_proto_multiple(proto_dir, mri_dir, subject, ['faces', 'bodies', 'places'], 
                                       ['OFA', 'EBA', 'OPA'], seed=42, split='train', hemi=hemi)


voxel_label = dataset.voxel_name
voxel_index = dataset.voxel_index

model = torch.load(os.path.join('./log_mri2proto_slim_multi_conv', subject, hemi, 'best_model.pt'))['model']
proto_weight = model['proto_vector.weight'].cpu().numpy()

6511 voxels selected and 7872 samples for train split


In [4]:
# -----------------------------
# Helper functions
# -----------------------------
def vector_to_view(vec):
    vec = np.asarray(vec, dtype=float)
    vec = vec / (np.linalg.norm(vec) + 1e-12)
    elev = np.degrees(np.arcsin(np.clip(vec[2], -1, 1)))
    azim = np.degrees(np.arctan2(vec[1], vec[0]))
    return (float(elev), float(azim))


def get_face_on_view(coords):
    """
    Camera view perpendicular to the best-fit plane of a flat mesh.
    """
    xyz = coords - coords.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(xyz, full_matrices=False)
    normal = vh[-1]
    if normal[2] < 0:
        normal = -normal
    return vector_to_view(normal)


def get_lateral_view(coords):
    """
    Approximate lateral view from mesh geometry.
    Left hemi -> look from negative x side
    Right hemi -> look from positive x side
    """
    x_sign = -1.0 if np.median(coords[:, 0]) < 0 else 1.0
    return vector_to_view(np.array([x_sign, 0.0, 0.0]))


def view_vector_from_elev_azim(elev, azim):
    elev_r = np.deg2rad(elev)
    azim_r = np.deg2rad(azim)
    return np.array([
        np.cos(elev_r) * np.cos(azim_r),
        np.cos(elev_r) * np.sin(azim_r),
        np.sin(elev_r)
    ], dtype=float)


def compute_roi_boundary_segments(coords, faces, vertex_mask,
                                  offset_scale=0.002, offset_vec=None):
    """
    Extract ROI boundary line segments from a triangular mesh.

    If offset_vec is given, lift all segments slightly along that vector
    (useful for flatmaps so boundaries appear in front of the patch).
    Otherwise use local triangle normals (better for 3D inflated surfaces).
    """
    coords = np.asarray(coords)
    faces = np.asarray(faces, dtype=np.int64)
    vertex_mask = np.asarray(vertex_mask, dtype=bool)

    mesh_diag = np.linalg.norm(np.ptp(coords, axis=0))
    offset = offset_scale * mesh_diag
    mesh_center = coords.mean(axis=0)

    if offset_vec is not None:
        offset_vec = np.asarray(offset_vec, dtype=float)
        offset_vec = offset_vec / (np.linalg.norm(offset_vec) + 1e-12)

    tri_mask = vertex_mask[faces]
    inside_count = tri_mask.sum(axis=1)
    boundary_face_idx = np.where((inside_count == 1) | (inside_count == 2))[0]

    segments = []

    for fi in boundary_face_idx:
        tri = faces[fi]
        tri_coords = coords[tri]
        tri_inside = tri_mask[fi]

        cut_points = []
        for i, j in ((0, 1), (1, 2), (2, 0)):
            if tri_inside[i] != tri_inside[j]:
                cut_points.append(0.5 * (tri_coords[i] + tri_coords[j]))

        if len(cut_points) != 2:
            continue

        p0, p1 = cut_points

        if offset_vec is not None:
            # flatmap: lift toward camera
            p0 = p0 + offset * offset_vec
            p1 = p1 + offset * offset_vec
        else:
            # 3D inflated surface: lift along local triangle normal
            normal = np.cross(
                tri_coords[1] - tri_coords[0],
                tri_coords[2] - tri_coords[0]
            )
            norm = np.linalg.norm(normal)
            if norm > 0:
                normal = normal / norm
                tri_center = tri_coords.mean(axis=0)
                if np.dot(normal, tri_center - mesh_center) < 0:
                    normal = -normal
                p0 = p0 + offset * normal
                p1 = p1 + offset * normal

        segments.append(np.vstack([p0, p1]))

    return segments


In [5]:
def plot_proto_map(proto_weight, voxel_index, voxel_label, proto_index=0, subject='subj01', hemi='lh', 
                   root_dir = "/mnt/data/UserData/shichen/data/NSD_dataset",
                   save_dir = None,
                   is_flatmap = True, boundary_linewidth = 3, boundary_alpha = 0.98):
    plt.close("all")
    
    # -----------------------------
    # Paths
    # -----------------------------
    data_dir = os.path.join(root_dir, subject)
    hemisphere = "left" if hemi == 'lh' else 'right'
    cur_save_dir = os.path.join(save_dir, subject, hemi)
    os.makedirs(cur_save_dir, exist_ok=True)
    
    voxel_indices = np.array(voxel_index)
    voxel_roi_names = np.array(voxel_label)
    proto_map = proto_weight[:, proto_index]
    
    # -----------------------------
    # ROI definitions
    # -----------------------------
    roi_info = {
        "EBA": "floc-bodies",
        "OFA": "floc-faces",
        "OPA": "floc-places",
        # "V1v": 'prf-visualrois',
        # "V2v": 'prf-visualrois',
        # "V3v": 'prf-visualrois',
        # "hV4": 'prf-visualrois',
        # "OWFA": "floc-words"
    }
    
    roi_list = list(roi_info.keys())
    
    # distinct publication-friendly colors
    roi_boundary_colors = {
        # "EBA":  "#CC79A7", # pink
        "EBA": "#AA00FF",
        "OFA":  "#FFD700",  # gold
        "OPA":  "#39FF14",  # neon green
        # "OWFA": "#00C853",  # vivid green
        # "V1v":  "#CC79A7", # pink
        # "V2v":  "#FFD700",  # gold
        # "V3v": "#1ABC9C",  # vivid green
        # "hV4":  "#39FF14",  # neon green
        
    }
    
    
    # -----------------------------
    # Load surface
    # -----------------------------
    fsaverage = datasets.fetch_surf_fsaverage("fsaverage")
    
    if not is_flatmap:
        #3D
        coords, faces = load_surf_mesh(fsaverage["infl_" + hemisphere])
    else:
        # flat map
        surf_mesh = fsaverage["flat_" + hemisphere]
        coords, faces = load_surf_mesh(surf_mesh)
    
    n_vertices = coords.shape[0]
    fsaverage_response = np.zeros(n_vertices)
    roi_vertex_masks = {}
    
    # -----------------------------
    # Process ROIs
    # -----------------------------
    for roi_name, roi_class in roi_info.items():
    
        challenge_roi_class = np.load(
            os.path.join(
                data_dir,
                "roi_masks",
                hemisphere[0] + "h." + roi_class + "_challenge_space.npy"
            )
        )
    
        fsaverage_roi_class = np.load(
            os.path.join(
                data_dir,
                "roi_masks",
                hemisphere[0] + "h." + roi_class + "_fsaverage_space.npy"
            )
        )
    
        roi_map = np.load(
            os.path.join(data_dir, "roi_masks", "mapping_" + roi_class + ".npy"),
            allow_pickle=True
        ).item()
    
        if roi_name not in roi_map.values():
            continue
    
        roi_mapping = list(roi_map.keys())[list(roi_map.values()).index(roi_name)]
        
        # save fsaverage ROI mask for boundary plotting
        roi_vertex_masks[roi_name] = (fsaverage_roi_class == roi_mapping)
    
        challenge_idx = np.where(challenge_roi_class == roi_mapping)[0]
        fsavg_idx = np.where(fsaverage_roi_class == roi_mapping)[0]
    
        vox_mask = voxel_roi_names == roi_name
        voxels_roi = voxel_indices[vox_mask]
        roi_proto_map = proto_map[vox_mask]
    
        if len(voxels_roi) == 0:
            continue
    
        # fast mapping
        order = np.argsort(challenge_idx)
        challenge_sorted = challenge_idx[order]
        fsavg_sorted = fsavg_idx[order]
    
        positions = np.searchsorted(challenge_sorted, voxels_roi)
    
        valid = (
            (positions < len(challenge_sorted)) &
            (challenge_sorted[positions] == voxels_roi)
        )
    
        fs_vertices = fsavg_sorted[positions[valid]]
        clusters_valid = roi_proto_map[valid]
        empty_mask = fsaverage_response[fs_vertices] == 0
        fsaverage_response[fs_vertices[empty_mask]] = clusters_valid[empty_mask]
    
    
    # -----------------------------
    # Plot
    # -----------------------------
    if not is_flatmap:
        # -----------------------------
        # 3D Plot (rolled back: no ROI boundaries)
        # -----------------------------
        view = plotting.view_surf(
            surf_mesh=fsaverage["infl_" + hemisphere],
            surf_map=fsaverage_response,
            bg_map=fsaverage["sulc_" + hemisphere],
            cmap="cold_hot",
            threshold=1e-14,
            colorbar=False
        )

        return view
    
    else:
        # -----------------------------
        # Flatmap plot + aligned ROI contours
        # -----------------------------
        plt.close("all")
    
        flat_view = get_face_on_view(coords)
    
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection="3d")
    
        # orthographic projection reduces flatmap distortion
        try:
            ax.set_proj_type("ortho")
        except Exception:
            pass
    
        plotting.plot_surf_stat_map(
            surf_mesh=surf_mesh,
            stat_map=fsaverage_response,
            bg_map=fsaverage["sulc_" + hemisphere],
            hemi=hemisphere,
            cmap="cold_hot",
            threshold=1e-14,
            colorbar=False,
            bg_on_data=True,
            # title=f"Prototype {proto_index}",
            view=flat_view,
            axes=ax,
            figure=fig
        )
    
        # re-apply exact camera after plotting
        ax.view_init(elev=flat_view[0], azim=flat_view[1])
    
        # draw ROI contours using nilearn's contour function
        valid_rois = []
        for roi_name in roi_list:
            if roi_name not in roi_vertex_masks:
                continue
    
            roi_mask = roi_vertex_masks[roi_name].astype(np.int32)
    
            # skip empty masks
            if roi_mask.sum() == 0:
                continue
    
            plotting.plot_surf_contours(
                surf_mesh=surf_mesh,
                roi_map=roi_mask,
                levels=[1],
                colors=[roi_boundary_colors[roi_name]],
                axes=ax,
                figure=fig,
                legend=False
            )
    
            valid_rois.append(roi_name)
    
        # # publication-style legend
        # if len(valid_rois) > 0:
        #     legend_handles = [
        #         Line2D(
        #             [0], [0],
        #             color=roi_boundary_colors[r],
        #             lw=2.4,
        #             label=r
        #         )
        #         for r in valid_rois
        #     ]
    
            # ax.legend(
            #     handles=legend_handles,
            #     loc="upper right",
            #     frameon=False,
            #     title="ROI boundaries",
            #     fontsize=11,
            #     title_fontsize=12
            # )
    
        # helps avoid extreme axis distortion
        ax.set_box_aspect((
            np.ptp(coords[:, 0]),
            np.ptp(coords[:, 1]),
            max(np.ptp(coords[:, 2]), 1e-3),
        ))
    
        plt.savefig(os.path.join(cur_save_dir, str(proto_index)+'.png'), bbox_inches='tight', dpi=800)